In [1]:
# Tool definition

%load_ext autoreload
%autoreload 2

from tool_geometries import*
import casadi as ca


In [ ]:
from tool_geometries import*

# Hiirt parameters
Dext = 200 # mm - external diameter
Dint = 150 # mm - internal diameter
H = 8.3 # mm - tooth height

# profile_parameters
alpha = 30 # degrees - profile angle
rho_fillet = 1 # mm - fillet radius
Pc = 0 # mm - profile crowning
rho_blade = (H/np.cos(alpha*np.pi/180)*0.5)**2/2/Pc # mm - blade curvature radius
if Pc == 0:
    rho_blade = 1e9 # mm - blade curvature radius for straight profile
Lc = 0 # mm - lengthwise crowning

# manufacturing errors
theta_error = 8*np.pi/180 # degrees
alpha_error = 8*np.pi/180 #
Xm_error = 2 # mm
Ym_error = 2 # mm
delta_error = 8*np.pi/180 # mm

profile_fun, normals_fun, u1, u2, point_height, s = hiirt_tool2D(H, alpha, s = None, rho_blade = rho_blade, rho_fillet = rho_fillet)
# cutter path for lengthwise crowning
G_parabolic, p_fun, tvec_fun, nvec_fun, bvec_fun = parabolic_path(Lc, alpha, Dext, Dint)

# tool-blank kinematics
Gcp = hiirt_kinematics(Dext, Dint, Lc, theta_error, alpha_error, Xm_error, Ym_error, delta_error, alpha, H)

# casadi symbolic variables
u = ca.SX.sym('u')
s = ca.SX.sym('s')

Gcp_expr = Gcp(s)
profile_expr = profile_fun(u)
normals_expr = normals_fun(u)
surf_expr = Gcp(s) @ ca.vertcat(0, profile_expr[0], -profile_expr[1]-Ym_error, 1)
surf_expr_left = Gcp(s) @ ca.vertcat(0, -profile_expr[0], -profile_expr[1]-Ym_error, 1)
surf_fun = ca.Function('surf_fun', [u, s], [surf_expr])
surf_fun_left = ca.Function('surf_fun_left', [u, s], [surf_expr_left])

3.1500001549720764
0.5000000026500002


C:\Users\egrab\AppData\Local\Temp\ipykernel_6884\3863494883.py:12: RuntimeWarning: divide by zero encountered in scalar divide
  rho_blade = (H/np.cos(alpha*np.pi/180)*0.5)**2/2/Pc # mm - blade curvature radius


In [3]:
# plot profile
s_num = np.linspace(-(Dext/2-Dint/2)/2-5, (Dext/2-Dint/2)/2+5, 200)
u_num = np.linspace(0, u2*2, 200)

profile_points = profile_fun(u_num.reshape(1,-1)).full()
profile_points[1,:] = -profile_points[1, :]  # flip y-axis to match the Hiirt convention
normals_points = normals_fun(u_num.reshape(1,-1)).full()
normals_points[1,:] = -normals_points[1, :]  # flip y-axis to match the Hiirt convention

# %matplotlib qt
# import matplotlib.pyplot as plt
# plt.figure(figsize=(10, 5))
# plt.plot(profile_points[0, :], profile_points[1, :], label='Rack Profile')
# # plt.quiver(profile_points[0, :], profile_points[1, :], normals_points[0, :], normals_points[1, :])
# plt.axis('equal')
# plt.xlim(0, 4)
# plt.show()


In [4]:
# build grid data points for the surface
S, U = np.meshgrid(s_num, u_num)
points = surf_fun(U.reshape(1, -1), S.reshape(1, -1))
points = points.full()

X = points[0, :].reshape(U.shape)
Y = points[1, :].reshape(U.shape)
Z = points[2, :].reshape(U.shape)

points_left = surf_fun_left(U.reshape(1, -1), S.reshape(1, -1))
points_left = points_left.full()
X_left = points_left[0, :].reshape(U.shape)
Y_left = points_left[1, :].reshape(U.shape)
Z_left = points_left[2, :].reshape(U.shape)

# X = np.concatenate((X, np.flip(X_left, axis  = 1)), axis=1)
# Y = np.concatenate((Y, np.flip(Y_left, axis  = 1)), axis=1)
# Z = np.concatenate((Z, np.flip(Z_left, axis  = 1)), axis=1)

X = np.concatenate((np.flip(X, axis = 0), X_left), axis=0) # np.flip(X, axis = 0)
Y = np.concatenate((np.flip(Y, axis = 0), Y_left), axis=0)
Z = np.concatenate((np.flip(Z, axis = 0), Z_left), axis=0)

# import easy_plot as ep


# fig = ep.Figure()
# surface = ep.surface(fig, X, Y, Z)
# fig.updateImage()
# fig.show()
print(np.full((3), 0))

[0 0 0]


In [5]:
# fit the data points with a nurbs surface
import nurbs as nrb
surf = nrb.Nurbs(knotsU = None, knotsV = None, degU = 2, degV = 2, control_points = None)

Q = {'X': X, 'Y': Y, 'Z': Z}
surf.fit(Q, 2, 2, [40, 20]) # control points [profile, lengthwise direction]
# surf.plot()

stp_writer = nrb.initialize_step_writer()
nurbs_OCC = nrb.Nurbs_to_STEPwriter(surf, stp_writer)

# write step to file
stp_writer.Write('test.step')



1

In [ ]:
# HIIRT blank parameters
num_teeth = 48
Dext_blank = Dext # mm - external diameter
Dint_blank = Dint # mm - internal diameter
H_base = 10 # mm - base height
H_master = 50 # mm - master height
crown_height = 15 # mm - height of the crowning
D_base = 250
D_hole = 50 # mm - hole diameter

In [7]:
from OCC.Core.gp import gp_Pnt, gp_Dir, gp_Ax1, gp_Vec
from OCC.Core.BRepBuilderAPI import BRepBuilderAPI_MakeEdge, BRepBuilderAPI_MakeWire, BRepBuilderAPI_MakeFace
from OCC.Core.BRepPrimAPI import BRepPrimAPI_MakeRevol
from OCC.Display.SimpleGui import init_display

# 1. Define profile points in the XZ plane
points = [
    gp_Pnt(D_base/2, 0, 0),
    gp_Pnt(D_base/2, 0, H_base),
    gp_Pnt(Dext_blank/2, 0, H_base),
    gp_Pnt(Dext_blank/2, 0, H_base + H_master + crown_height),
    gp_Pnt(Dint_blank/2, 0, H_base + H_master + crown_height),
    gp_Pnt(Dint_blank/2, 0, H_base + H_master),
    gp_Pnt(D_hole/2, 0, H_base + H_master),
    gp_Pnt(D_hole/2, 0, 0),
    gp_Pnt(D_base/2, 0, 0)
    ]

# 2. Create edges between consecutive points
edges = [BRepBuilderAPI_MakeEdge(points[i], points[i + 1]).Edge() for i in range(len(points) - 1)]

# 3. Create a wire from the edges
wire_builder = BRepBuilderAPI_MakeWire()
for edge in edges:
    wire_builder.Add(edge)
profile_wire = wire_builder.Wire()
face = BRepBuilderAPI_MakeFace(profile_wire).Face()

# 4. Define revolution axis (Z-axis)
revolution_axis = gp_Ax1(gp_Pnt(0, 0, 0), gp_Dir(0, 0, 1))

# 5. Revolve profile around Z-axis (360 degrees)
revolved_shape = BRepPrimAPI_MakeRevol(face, revolution_axis).Shape()



In [8]:
# perform boolean operation to subtract the internal cylinder
from OCC.Core.BRepAlgoAPI import BRepAlgoAPI_Cut, BRepAlgoAPI_Fuse
from OCC.Core.BRepPrimAPI import BRepPrimAPI_MakePrism
from OCC.Core.BRepBuilderAPI import BRepBuilderAPI_MakeFace
from OCC.Core.ShapeFix import ShapeFix_Shape

nurbs_Face = BRepBuilderAPI_MakeFace(nurbs_OCC, 1e-6).Face()
extrude_vec = gp_Vec(0, 0, 10)  # or another direction to pierce the blank
cutter_solid = BRepPrimAPI_MakePrism(nurbs_Face, extrude_vec).Shape()
cut = BRepAlgoAPI_Cut(revolved_shape, cutter_solid)

if not cut.IsDone():
    raise RuntimeError("Boolean cut failed.")

final_result = cut.Shape()
# # result_shape: output from BRepAlgoAPI_Cut
# fixer = ShapeFix_Shape(final_result)
# fixer.Perform()

# # Optional: enforce validity checks
# fixer.SetPrecision(1e-6)
# fixer.SetMaxTolerance(1e-4)

# Cleaned shape
# cleaned_shape = fixer.Shape()

In [9]:
# Display the stuff
from OCC.Display.SimpleGui import init_display
display, start_display, add_menu, add_function_to_menu = init_display()
display.DisplayShape(nurbs_OCC)
# display.DisplayShape(revolved_shape, update=True)
display.DisplayShape(final_result, update=True)
start_display()

pyqt5 backend - Qt version 5.15.8
INFO:OCC.Display.qtDisplay:key: code 16777216 not mapped to any function


In [16]:
from OCC.Core.STEPControl import STEPControl_Writer, STEPControl_AsIs
from OCC.Core.Interface import Interface_Static

step_writer_body = STEPControl_Writer()
Interface_Static.SetCVal("write.step.schema", "AP203")
step_writer_body.Transfer(final_result, STEPControl_AsIs)
step_writer_body.Write('HIRTH.step')

1